# CrewAI Customer Retention Crew

**Project:** Multi-Agent Customer Complaint Resolution & Retention Pipeline  
**Framework:** CrewAI (multi-agent orchestration)  
**LLM Backend:** Ollama (local, qwen3 model)  

---

## Overview

This notebook builds a **4-agent customer retention crew** that processes customer complaints end-to-end — from root cause analysis through churn risk scoring, resolution drafting, and account summarization.

### The Agent Roster

| # | Agent | Role | Deliverable |
|---|-------|------|------------|
| 1 | **Complaint Analyst** | Root Cause Investigator | Structured complaint analysis — category, severity, root cause, sentiment |
| 2 | **Churn Risk Agent** | Retention Risk Scorer | Churn risk assessment — risk level, contributing factors, retention urgency |
| 3 | **Resolution Drafter** | Response & Action Planner | Resolution package — customer response, internal actions, compensation |
| 4 | **Account Summarizer** | 360° Account Reporter | Account summary — complaint history, risk profile, recommended next steps |

### Collaboration Flow — How Agents Build on Each Other

```
Customer Complaint (raw ticket)
          │
          ▼
┌─────────────────────┐
│  Complaint Analyst  │  DIAGNOSE — "What happened and why?"
└──────────┬──────────┘
           │ Root cause analysis (category, severity, emotion)
           ▼
┌─────────────────────┐
│   Churn Risk Agent  │  ASSESS — "Should we worry about losing this customer?"
└──────────┬──────────┘
           │ Risk score + contributing factors + urgency
           ▼
┌─────────────────────┐
│ Resolution Drafter  │  RESPOND — "What do we say and do?"
└──────────┬──────────┘
           │ Customer response + internal actions + compensation
           ▼
┌─────────────────────┐
│ Account Summarizer  │  DOCUMENT — "What's the full picture for this account?"
└─────────────────────┘
           │
           ▼
  Complete retention case file
```

### Understanding Collaboration Flow

**Collaboration flow** answers the question: *"How does one agent's output shape the next agent's decisions?"*

In this crew, each agent doesn't just receive prior outputs as passive context — each agent **changes its behavior** based on what the upstream agent found:

| Upstream Finding | Downstream Behavior Change |
|-----------------|---------------------------|
| Complaint Analyst finds **billing error** | Churn Risk Agent lowers risk (fixable problem) |
| Complaint Analyst finds **repeated issue** | Churn Risk Agent raises risk (pattern of failure) |
| Churn Risk Agent scores **HIGH** | Resolution Drafter escalates compensation and tone |
| Churn Risk Agent scores **LOW** | Resolution Drafter uses standard response |
| Resolution Drafter proposes **account credit** | Account Summarizer flags financial impact |
| Resolution Drafter proposes **escalation** | Account Summarizer recommends management review |

This is **adaptive collaboration** — each agent doesn't follow a fixed script but adjusts its output based on what it learns from the prior agent. The collaboration flow creates emergent intelligence that no single agent could produce.

## 1. Environment Setup

In [ ]:
# Install dependencies
!pip install -q crewai crewai-tools langchain-ollama

In [ ]:
import os
import json
import textwrap
from datetime import datetime

from crewai import Agent, Task, Crew, Process
from crewai import LLM

print("CrewAI imports successful")
print(f"Timestamp: {datetime.now().isoformat()}")

## 2. LLM Configuration

All agents share the same local Ollama model. The collaboration flow emerges from how each agent's backstory instructs it to interpret and build on prior outputs.

In [ ]:
llm = LLM(
    model="ollama/qwen3",
    base_url="http://localhost:11434",
    temperature=0.7,
)

print(f"LLM configured: {llm.model}")

## 3. Define the Agents

### Designing Agents for Collaboration

When agents collaborate (rather than work independently), each agent's design must account for **what it receives** and **what it passes forward**:

| Agent | Receives From | Must Produce For | Collaboration Contract |
|-------|--------------|-----------------|----------------------|
| Complaint Analyst | Raw complaint only | Churn Risk Agent | Classification + severity + emotion = risk inputs |
| Churn Risk Agent | Analyst's diagnosis | Resolution Drafter | Risk level determines response intensity |
| Resolution Drafter | Diagnosis + risk score | Account Summarizer | Actions taken become account history |
| Account Summarizer | All three prior outputs | End consumer (support team) | Unified narrative with forward-looking recommendations |

**Key design principle:** Each agent's output format must contain exactly the fields the downstream agent needs. The Complaint Analyst doesn't just describe the problem — it classifies severity on a scale the Churn Risk Agent knows how to interpret.

### 3.1 Complaint Analyst Agent

**Collaboration role:** The **diagnostic foundation**. Every downstream agent's behavior is shaped by the Complaint Analyst's classification. If the analyst gets the root cause wrong, the entire pipeline produces a misaligned response.

**Collaboration contract:** Must output structured fields (category, severity, root cause, sentiment, repeat flag) that the Churn Risk Agent directly uses as scoring inputs.

In [ ]:
complaint_analyst = Agent(
    role="Customer Complaint Analyst & Root Cause Investigator",
    goal=(
        "Analyze the customer complaint to identify the root cause, classify the "
        "complaint category, assess emotional intensity, determine severity level, "
        "and flag whether this is a first-time or repeat issue. Produce a structured "
        "diagnostic report that downstream agents can act on."
    ),
    backstory=(
        "You are a senior customer experience analyst with 9 years of experience at "
        "high-volume SaaS companies (Zendesk, Intercom, Freshdesk). You have analyzed "
        "50,000+ complaint tickets and developed a diagnostic framework that support teams "
        "rely on for triage: (1) Complaint Category — Billing/Payment, Product Bug, Feature "
        "Gap, Service Quality, Onboarding Friction, Data/Privacy, Performance/Downtime, "
        "Account Access, (2) Severity — S1-Critical (service unusable, data loss), S2-High "
        "(major feature broken, revenue impact), S3-Medium (degraded experience, workaround "
        "exists), S4-Low (cosmetic, minor inconvenience), (3) Root Cause — the actual "
        "underlying issue, not just the symptom described, (4) Emotional Intensity — Calm "
        "(factual report), Frustrated (expressed dissatisfaction), Angry (demanded action, "
        "threats to leave), Escalated (already contacted management, posted publicly), "
        "(5) Repeat Flag — is this the first time, or has the customer reported this before? "
        "(6) Customer Effort Score — how much effort has the customer already spent trying to "
        "resolve this? Your analysis always separates the symptom (what the customer reports) "
        "from the root cause (what actually went wrong), because the resolution must address "
        "the root cause, not just the symptom."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {complaint_analyst.role}")

### 3.2 Churn Risk Agent

**Collaboration role:** The **risk calibrator**. This agent transforms the Complaint Analyst's diagnosis into a risk score that determines how aggressively the Resolution Drafter responds.

**Collaboration contract:** Reads the analyst's severity, repeat flag, and emotional intensity as direct inputs to its risk model. Must output a clear risk tier (Critical/High/Medium/Low) and a retention urgency level that the Resolution Drafter maps to response intensity.

**Adaptive behavior example:**
- If analyst found S1-Critical + Repeat + Angry → Churn Risk outputs CRITICAL risk
- If analyst found S4-Low + First-time + Calm → Churn Risk outputs LOW risk
- Same agent, same model, completely different output driven by upstream context

In [ ]:
churn_risk_agent = Agent(
    role="Customer Churn Risk Assessor & Retention Strategist",
    goal=(
        "Using the Complaint Analyst's diagnostic report, assess the customer's "
        "churn risk level. Score the risk based on complaint severity, emotional "
        "intensity, account tenure, repeat issue history, and competitive alternatives. "
        "Produce a risk tier (Critical/High/Medium/Low) with contributing factors and "
        "a retention urgency classification."
    ),
    backstory=(
        "You are a customer retention specialist and churn prediction expert with 7 years "
        "of experience at subscription SaaS companies. You have built churn prediction "
        "models and designed retention playbooks that reduced churn by 35%. Your risk "
        "assessment framework: (1) Churn Signal Scoring — assign weighted points across: "
        "complaint severity (S1=40pts, S2=30, S3=15, S4=5), emotional intensity (Escalated=30, "
        "Angry=20, Frustrated=10, Calm=0), repeat issue (repeat=25pts, first-time=0), "
        "customer effort already spent (high=20, medium=10, low=0), (2) Risk Tier — Critical "
        "(70+ points: immediate intervention), High (50-69: proactive outreach within 24h), "
        "Medium (25-49: standard resolution), Low (0-24: routine handling), (3) Contributing "
        "Factors — list the top 3 factors driving the risk score, (4) Retention Urgency — "
        "Immediate (call today), Urgent (respond within 4h), Standard (next business day), "
        "Routine (queue), (5) Competitive Exposure — given this complaint type, how easy is it "
        "for the customer to switch to a competitor? You also flag Red Flags: contract renewal "
        "approaching, recent downgrade, decreased usage, or public complaints on social media."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {churn_risk_agent.role}")

### 3.3 Resolution Drafter Agent

**Collaboration role:** The **response calibrator**. This agent reads both the diagnosis AND the risk score to craft a response whose tone, compensation, and urgency precisely match the situation.

**Collaboration contract:** The risk tier directly maps to response intensity:

| Risk Tier | Response Intensity | Compensation Range | Tone |
|-----------|-------------------|-------------------|------|
| Critical | Executive involvement, immediate call | Full refund + extra credits | Empathetic, ownership-taking |
| High | Senior agent, proactive outreach | Partial refund or credits | Warm, solution-focused |
| Medium | Standard agent, quality response | Goodwill gesture | Professional, helpful |
| Low | Template-enhanced response | None needed | Friendly, efficient |

**This is the most visible collaboration point** — the Resolution Drafter's output quality directly depends on the quality of both upstream agents.

In [ ]:
resolution_drafter = Agent(
    role="Customer Resolution Specialist & Response Architect",
    goal=(
        "Draft a complete resolution package: a customer-facing response email, "
        "internal action items for the support team, and a compensation/goodwill "
        "recommendation calibrated to the churn risk level. The response tone and "
        "generosity must match the risk assessment — critical risk gets executive "
        "attention, low risk gets efficient resolution."
    ),
    backstory=(
        "You are a customer resolution architect with 8 years of experience designing "
        "response frameworks for SaaS support teams. You have drafted 10,000+ customer "
        "responses and your templates are used by a 50-person support team. Your approach "
        "is risk-calibrated: (1) Response Tone Mapping — match tone to emotional intensity: "
        "Calm → Professional and efficient, Frustrated → Empathetic and solution-focused, "
        "Angry → Ownership-taking and proactive, Escalated → Executive-level with personal "
        "commitment, (2) Resolution Components — every response has: Acknowledgment (validate "
        "the experience), Root Cause (explain what happened — from the complaint analysis), "
        "Resolution (specific fix or workaround), Prevention (what we're doing so it won't "
        "recur), Compensation (if warranted by risk level), Follow-up (when and how we'll "
        "check back), (3) Compensation Matrix — Critical risk: full refund + service credits + "
        "dedicated account manager, High risk: partial refund or significant credits + priority "
        "support, Medium risk: goodwill credit or feature upgrade, Low risk: sincere apology + "
        "educational resources, (4) Internal Actions — create specific, assigned tickets for "
        "engineering, product, billing, or account management. You never draft a response "
        "without also specifying the internal actions needed to actually fix the problem."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {resolution_drafter.role}")

### 3.4 Account Summarizer Agent

**Collaboration role:** The **context aggregator**. This agent sees ALL prior outputs and produces a unified account record — think of it as the "case file" that a support manager reads to understand the full situation at a glance.

**Collaboration contract:** Consumes the complaint analysis, risk assessment, AND resolution package to produce a forward-looking summary. This is the only agent whose output is designed for archival — it becomes part of the customer's permanent record.

**Why a separate summarizer?** Without it, the account record is scattered across three separate outputs. The summarizer produces one document that answers: "What happened? How serious is it? What did we do? What happens next?" 

In [ ]:
account_summarizer = Agent(
    role="Customer Account Analyst & Case Summary Specialist",
    goal=(
        "Produce a 360-degree account summary that integrates the complaint analysis, "
        "churn risk assessment, and resolution actions into a single case file. Include "
        "a health score, complaint pattern analysis, financial impact of the resolution, "
        "and forward-looking retention recommendations."
    ),
    backstory=(
        "You are a customer success operations analyst with 6 years of experience managing "
        "account health dashboards and case documentation for enterprise SaaS companies. "
        "You write the account summaries that CSMs, account executives, and VPs of Customer "
        "Success read before every renewal conversation and QBR. Your summary framework: "
        "(1) Case Snapshot — one-paragraph executive summary of the complaint, risk, and "
        "resolution in plain language, (2) Account Health Score — rate the account: Healthy "
        "(no concerns), At Risk (monitor closely), Critical (intervention needed), based on "
        "the cumulative picture, (3) Complaint Pattern — is this an isolated incident or part "
        "of a pattern? What trends does the account show? (4) Financial Impact — quantify the "
        "cost of the resolution: credits issued, refund amount, support hours spent, and "
        "revenue at risk if the customer churns, (5) Resolution Status — what was done, what's "
        "pending, and who owns each action item? (6) Forward-Looking Recommendations — 3-5 "
        "specific actions to strengthen the relationship: proactive outreach schedule, feature "
        "adoption guidance, executive sponsor assignment, renewal preparation, (7) Account Tags — "
        "standardized labels for CRM tagging (e.g., #billing-issue, #high-churn-risk, "
        "#proactive-outreach-needed). Your summaries are designed to be skimmable in 60 seconds "
        "and to serve as the single source of truth for the account."
    ),
    llm=llm,
    verbose=True,
    allow_delegation=False,
)

print(f"Agent created: {account_summarizer.role}")

### Agent Roster — Collaboration Map

In [ ]:
agents = [complaint_analyst, churn_risk_agent, resolution_drafter, account_summarizer]
collab_roles = [
    ("Diagnose", "Structured diagnosis", "feeds → risk scoring inputs"),
    ("Assess", "Risk tier + urgency", "feeds → response calibration"),
    ("Respond", "Resolution package", "feeds → account documentation"),
    ("Document", "360° case file", "feeds → CRM + support team"),
]

print("=" * 70)
print("CUSTOMER RETENTION CREW — COLLABORATION MAP")
print("=" * 70)
for i, (agent, (phase, output, flow)) in enumerate(zip(agents, collab_roles), 1):
    print(f"\n[{i}] {agent.role}")
    print(f"    Phase: {phase}")
    print(f"    Output: {output}")
    print(f"    Collaboration: {flow}")
print("\n" + "=" * 70)
print("\nFlow: DIAGNOSE → ASSESS → RESPOND → DOCUMENT")
print("Each agent adapts its behavior based on upstream findings.")

## 4. Define Tasks & Collaboration Pipeline

### Collaboration Flow in Task Design

Each task explicitly references the upstream agent's output format, creating a **collaboration contract**:

```
Complaint Analyst outputs:         Churn Risk Agent reads:
├── Category (enum)           →    ├── Category → maps to churn correlation
├── Severity (S1-S4)          →    ├── Severity → weighted risk points
├── Root Cause (free text)         ├── Root Cause → fixability assessment
├── Emotional Intensity (enum) →   ├── Emotion → urgency modifier
├── Repeat Flag (bool)        →    ├── Repeat → major risk multiplier
└── Customer Effort (H/M/L)   →    └── Effort → frustration accumulation
```

**This is what makes it collaboration, not just sequential execution.** The Churn Risk Agent doesn't just "see" the complaint analysis — it uses specific fields from it as scoring inputs. The task description tells the agent exactly which upstream fields to consume and how to weight them.

### 4.1 Configure the Customer Complaint

In [ ]:
# =====================================================
# CONFIGURE YOUR CUSTOMER COMPLAINT HERE
# =====================================================
COMPLAINT = {
    "ticket_id": "SUP-2025-8847",
    "customer": "Sarah Chen",
    "company": "NovaTech Solutions",
    "plan": "Pro Plan ($499/mo)",
    "tenure": "14 months",
    "mrr": 499,
    "previous_tickets": 2,
    "last_ticket_topic": "Slow dashboard loading (3 months ago, resolved)",
    "complaint_text": (
        "I've been a customer for over a year and I'm really frustrated. Our team "
        "relies on your reporting dashboards for our Monday morning exec meetings, "
        "and for the THIRD time this quarter, the dashboards failed to load during "
        "our meeting. We looked unprepared in front of our VP. I've reported this "
        "before and was told it was fixed, but clearly it wasn't. Our contract is "
        "up for renewal in 6 weeks and honestly I'm evaluating Datadog and Grafana "
        "as alternatives. I need to know what you're going to do about this because "
        "my team has lost confidence in the platform."
    ),
    "channel": "Email (support@company.com)",
    "timestamp": "Monday 8:47 AM",
}
# =====================================================

print("CUSTOMER COMPLAINT")
print("=" * 60)
print(f"  Ticket ID:   {COMPLAINT['ticket_id']}")
print(f"  Customer:    {COMPLAINT['customer']} ({COMPLAINT['company']})")
print(f"  Plan:        {COMPLAINT['plan']}")
print(f"  Tenure:      {COMPLAINT['tenure']}")
print(f"  MRR:         ${COMPLAINT['mrr']}/mo")
print(f"  Prior Tickets: {COMPLAINT['previous_tickets']}")
print(f"  Channel:     {COMPLAINT['channel']}")
print(f"  Timestamp:   {COMPLAINT['timestamp']}")
print(f"\nComplaint Text:")
print(f"  {COMPLAINT['complaint_text']}")

In [ ]:
# Build the ticket context string
ticket_context = (
    f"=== SUPPORT TICKET ===\n"
    f"Ticket ID: {COMPLAINT['ticket_id']}\n"
    f"Customer: {COMPLAINT['customer']}\n"
    f"Company: {COMPLAINT['company']}\n"
    f"Plan: {COMPLAINT['plan']}\n"
    f"Tenure: {COMPLAINT['tenure']}\n"
    f"MRR: ${COMPLAINT['mrr']}/mo\n"
    f"Previous Tickets: {COMPLAINT['previous_tickets']} "
    f"(last: {COMPLAINT['last_ticket_topic']})\n"
    f"Channel: {COMPLAINT['channel']}\n"
    f"Timestamp: {COMPLAINT['timestamp']}\n\n"
    f"=== COMPLAINT TEXT ===\n"
    f"{COMPLAINT['complaint_text']}"
)
print(ticket_context)

### 4.2 Task 1 — Complaint Analysis (DIAGNOSE)

**Collaboration phase:** First in the chain. No upstream context — works from the raw complaint only.

**Downstream contract:** Must produce structured fields (category, severity, root cause, emotion, repeat flag, effort score) that the Churn Risk Agent consumes as direct scoring inputs.

In [ ]:
task_analyze = Task(
    description=(
        f"Analyze this customer complaint:\n\n{ticket_context}\n\n"
        "Produce a STRUCTURED diagnostic report with these exact fields:\n\n"
        "1. **Complaint Category**: Choose ONE primary category:\n"
        "   Billing/Payment | Product Bug | Feature Gap | Service Quality | "
        "   Onboarding Friction | Data/Privacy | Performance/Downtime | Account Access\n\n"
        "2. **Secondary Category** (if applicable)\n\n"
        "3. **Severity Level**: S1 (Critical — service unusable) / S2 (High — major "
        "   impact) / S3 (Medium — degraded, workaround exists) / S4 (Low — minor)\n\n"
        "4. **Root Cause Analysis**:\n"
        "   - SYMPTOM: What the customer is experiencing\n"
        "   - ROOT CAUSE: The underlying technical or process failure\n"
        "   - WHY IT RECURRED (if repeat): Why the previous fix didn't hold\n\n"
        "5. **Emotional Intensity**: Calm / Frustrated / Angry / Escalated\n"
        "   - Key emotional phrases from the complaint\n"
        "   - What the customer actually wants (stated or implied)\n\n"
        "6. **Repeat Issue Flag**: First-time / Repeat\n"
        "   - If repeat: how many prior occurrences and when\n\n"
        "7. **Customer Effort Score**: Low / Medium / High\n"
        "   - How much effort has the customer spent trying to resolve this?\n\n"
        "8. **Business Impact**: What is the customer's business impact?\n"
        "   - Internal (team productivity, credibility) and external (customer-facing)\n\n"
        "IMPORTANT: Use the exact field names and category values above. Downstream "
        "agents depend on this structure for scoring and response calibration.\n"
    ),
    expected_output=(
        "A structured complaint diagnostic with category, severity (S1-S4), root "
        "cause (symptom vs. root cause), emotional intensity, repeat flag, customer "
        "effort score, and business impact assessment."
    ),
    agent=complaint_analyst,
)

print(f"Task created: Complaint Analysis -> {task_analyze.agent.role}")

### 4.3 Task 2 — Churn Risk Assessment (ASSESS)

**Collaboration phase:** Receives the Complaint Analyst's structured diagnosis and transforms it into a risk score.

**Adaptive behavior:** The risk assessment changes entirely based on upstream findings:
- A billing error from a calm first-time reporter → LOW risk
- A recurring performance issue from an angry customer mentioning competitors → CRITICAL risk

**Downstream contract:** The risk tier directly determines how aggressive the Resolution Drafter's response will be.

In [ ]:
task_risk = Task(
    description=(
        "Using the Complaint Analyst's diagnostic report, assess this customer's "
        "churn risk.\n\n"
        f"Account Context: {COMPLAINT['customer']} at {COMPLAINT['company']}, "
        f"${COMPLAINT['mrr']}/mo, {COMPLAINT['tenure']} tenure, "
        f"renewal in 6 weeks.\n\n"
        "Apply the risk scoring framework:\n"
        "1. **Churn Signal Scoring** — assign points:\n"
        "   - Complaint severity: S1=40, S2=30, S3=15, S4=5\n"
        "   - Emotional intensity: Escalated=30, Angry=20, Frustrated=10, Calm=0\n"
        "   - Repeat issue: Repeat=25, First-time=0\n"
        "   - Customer effort: High=20, Medium=10, Low=0\n"
        "   - Competitive mention: Yes=15, No=0\n"
        "   - Renewal proximity (<90 days): Yes=10, No=0\n\n"
        "2. **Risk Tier**: Calculate total score and classify:\n"
        "   - Critical (70+): Immediate executive intervention\n"
        "   - High (50-69): Senior agent, proactive outreach within 24h\n"
        "   - Medium (25-49): Standard resolution with quality focus\n"
        "   - Low (0-24): Routine handling\n\n"
        "3. **Contributing Factors**: List the top 3 factors driving the score\n\n"
        "4. **Retention Urgency**: Immediate / Urgent / Standard / Routine\n\n"
        "5. **Red Flags**: Check for:\n"
        "   - Contract renewal approaching\n"
        "   - Competitor mentions\n"
        "   - Decreased usage pattern\n"
        "   - Public complaints (social media, review sites)\n"
        "   - Previous downgrades\n\n"
        "6. **Competitive Exposure**: How easy is it for this customer to switch?\n"
        "   Rate: High / Medium / Low with reasoning\n\n"
        "7. **Revenue at Risk**: ${COMPLAINT['mrr']}/mo × risk probability estimate\n\n"
        "IMPORTANT: Show your point calculation explicitly so downstream agents "
        "can see exactly why the risk is scored at this level.\n"
    ),
    expected_output=(
        "A churn risk assessment with explicit point calculation, risk tier "
        "(Critical/High/Medium/Low), top 3 contributing factors, retention "
        "urgency, red flags, competitive exposure, and revenue at risk."
    ),
    agent=churn_risk_agent,
)

print(f"Task created: Churn Risk -> {task_risk.agent.role}")

### 4.4 Task 3 — Resolution Drafting (RESPOND)

**Collaboration phase:** The most collaboration-dependent task. The Resolution Drafter must simultaneously:
1. Address the **root cause** identified by the Complaint Analyst (not just the symptom)
2. Match the **response intensity** to the Churn Risk Agent's risk tier
3. Calibrate **tone** to the customer's emotional state

**Adaptive collaboration in action:** Compare what this agent would produce for the same complaint at different risk levels:
- CRITICAL risk → Executive-signed response, full refund, dedicated account manager, same-day call
- LOW risk → Professional template, acknowledgment, timeline for fix, no compensation

In [ ]:
task_resolution = Task(
    description=(
        "Draft a complete resolution package using BOTH the complaint analysis "
        "AND the churn risk assessment.\n\n"
        f"Customer: {COMPLAINT['customer']} at {COMPLAINT['company']}\n"
        f"MRR: ${COMPLAINT['mrr']}/mo | Tenure: {COMPLAINT['tenure']}\n\n"
        "Your resolution MUST be calibrated to the risk tier from the Churn Risk Agent:\n"
        "- Critical risk → executive involvement, immediate call, maximum compensation\n"
        "- High risk → senior agent, proactive outreach, significant compensation\n"
        "- Medium risk → quality response, goodwill gesture\n"
        "- Low risk → efficient resolution, standard handling\n\n"
        "Produce THREE deliverables:\n\n"
        "**A. Customer-Facing Response Email**:\n"
        "   - Acknowledgment (validate their experience — reference specifics from complaint)\n"
        "   - Root cause explanation (from analyst's diagnosis — translate to customer language)\n"
        "   - Specific resolution steps (what we're doing to fix it)\n"
        "   - Prevention commitment (what changes prevent recurrence)\n"
        "   - Compensation (calibrated to risk tier — specify exact amounts/credits)\n"
        "   - Follow-up plan (when and how we'll check back)\n"
        "   - Personal commitment (appropriate to risk level)\n\n"
        "**B. Internal Action Items** (for support team):\n"
        "   - Engineering ticket(s) with priority level\n"
        "   - Account management follow-up schedule\n"
        "   - Escalation path if resolution doesn't hold\n"
        "   - Monitoring setup (how to proactively detect recurrence)\n\n"
        "**C. Compensation Recommendation**:\n"
        "   - What to offer (credit, refund, upgrade, dedicated support)\n"
        "   - Dollar amount or equivalent\n"
        "   - Justification (tied to risk tier and customer value)\n"
        "   - Approval required? (if above standard authority)\n"
    ),
    expected_output=(
        "A resolution package with: (A) customer-facing email calibrated to risk "
        "tier, (B) internal action items with owners and priorities, (C) compensation "
        "recommendation with dollar amount and justification."
    ),
    agent=resolution_drafter,
)

print(f"Task created: Resolution -> {task_resolution.agent.role}")

### 4.5 Task 4 — Account Summary (DOCUMENT)

**Collaboration phase:** Final agent — sees ALL three prior outputs and produces a unified case file.

**Collaboration dependency:** This is the most context-rich agent. It must:
- Synthesize the diagnosis (from analyst)
- Reference the risk score (from risk agent)
- Document the resolution (from drafter)
- Add forward-looking recommendations that none of the other agents produced

In [ ]:
task_summary = Task(
    description=(
        "Produce a 360-degree account summary that integrates ALL prior outputs "
        "(complaint analysis, risk assessment, resolution package) into a single "
        "case file.\n\n"
        f"Account: {COMPLAINT['customer']} at {COMPLAINT['company']}\n"
        f"Ticket: {COMPLAINT['ticket_id']}\n\n"
        "Your account summary MUST include:\n\n"
        "1. **Case Snapshot** (3-4 sentences):\n"
        "   Plain-language summary a VP could read in 15 seconds.\n"
        "   Include: what happened, how serious it is, what we're doing about it.\n\n"
        "2. **Account Health Score**: Healthy / At Risk / Critical\n"
        "   Based on cumulative picture (not just this ticket).\n\n"
        "3. **Complaint Pattern Analysis**:\n"
        "   - Is this isolated or part of a trend?\n"
        "   - What does the ticket history tell us?\n"
        "   - Pattern classification: Isolated / Recurring / Escalating\n\n"
        "4. **Financial Impact Summary**:\n"
        "   - Revenue at risk (MRR × months remaining in contract)\n"
        "   - Cost of resolution (credits, refunds, support hours)\n"
        "   - Net retention impact\n\n"
        "5. **Resolution Status**:\n"
        "   - Actions completed\n"
        "   - Actions pending (with owners and deadlines)\n"
        "   - Success criteria (how do we know the fix worked?)\n\n"
        "6. **Forward-Looking Recommendations** (3-5 specific actions):\n"
        "   - Proactive outreach schedule\n"
        "   - Feature adoption opportunities\n"
        "   - Renewal preparation steps\n"
        "   - Relationship strengthening tactics\n\n"
        "7. **CRM Tags**: Standardized labels for the account record\n"
        "   (e.g., #performance-issue, #high-churn-risk, #renewal-at-risk, "
        "   #proactive-outreach-needed)\n\n"
        "FORMAT: Skimmable in 60 seconds. Use headers, bullets, and bold for key "
        "findings. This summary becomes the permanent account record.\n"
    ),
    expected_output=(
        "A 360-degree account summary with case snapshot, health score, complaint "
        "pattern analysis, financial impact, resolution status, forward-looking "
        "recommendations, and CRM tags."
    ),
    agent=account_summarizer,
)

print(f"Task created: Account Summary -> {task_summary.agent.role}")

### Collaboration Flow Visualization

In [ ]:
tasks = [task_analyze, task_risk, task_resolution, task_summary]
task_names = ["Complaint Analysis", "Churn Risk Assessment", "Resolution Package", "Account Summary"]
phases = ["DIAGNOSE", "ASSESS", "RESPOND", "DOCUMENT"]
collab_info = [
    "→ category, severity, emotion, repeat flag feed into risk scoring",
    "→ risk tier determines response intensity and compensation level",
    "→ actions taken + compensation become part of account record",
    "→ unified case file for CRM, CSMs, and renewal team",
]

print("COLLABORATION FLOW — CUSTOMER RETENTION PIPELINE")
print("=" * 70)
for i, (phase, name, task, collab) in enumerate(zip(phases, task_names, tasks, collab_info)):
    print(f"  [{phase}] {name}")
    print(f"    Agent: {task.agent.role.split('&')[0].strip()}")
    print(f"    Collaboration: {collab}")
    if i < len(tasks) - 1:
        print(f"    {'':6}│")
        print(f"    {'':6}│  upstream output shapes downstream behavior ↓")
        print(f"    {'':6}│")
print("=" * 70)

## 5. Assemble and Run the Crew

### Why Sequential Process Fits Collaboration

Collaboration requires sequential execution because each agent's output is an **input** to the next agent's decision-making:

| If you ran in parallel... | What goes wrong |
|---------------------------|----------------|
| Risk Agent without diagnosis | Can't score — no severity, no emotion, no repeat flag |
| Resolution Drafter without risk tier | Can't calibrate tone — doesn't know if it's critical or routine |
| Account Summarizer without resolution | Can't document — doesn't know what actions were taken |

**Key insight:** Collaboration flow is inherently sequential when agents have data dependencies. Parallel execution only works for independent specialists (like in competitive intelligence).

In [ ]:
crew = Crew(
    agents=agents,
    tasks=tasks,
    process=Process.sequential,
    verbose=True,
    memory=False,
)

print(f"Crew assembled: {len(crew.agents)} agents, {len(crew.tasks)} tasks")
print(f"Process: {crew.process}")

In [ ]:
# Execute the retention pipeline
print("=" * 70)
print(f"LAUNCHING CUSTOMER RETENTION CREW — {datetime.now().strftime('%H:%M:%S')}")
print(f"Ticket: {COMPLAINT['ticket_id']}")
print(f"Customer: {COMPLAINT['customer']} at {COMPLAINT['company']}")
print(f"Flow: DIAGNOSE -> ASSESS -> RESPOND -> DOCUMENT")
print("=" * 70)

result = crew.kickoff()

print("\n" + "=" * 70)
print(f"CREW COMPLETED — {datetime.now().strftime('%H:%M:%S')}")
print("=" * 70)

## 6. Analyze Results

### 6.1 Final Output — Account Summary

In [ ]:
print("=" * 70)
print(f"ACCOUNT SUMMARY — {COMPLAINT['customer']} @ {COMPLAINT['company']}")
print("=" * 70)
print(result.raw)

### 6.2 Collaboration Trace — Phase-by-Phase Outputs

In [ ]:
for phase, name, task in zip(phases, task_names, tasks):
    print("\n" + "=" * 70)
    print(f"[{phase}] {name}")
    print(f"Agent: {task.agent.role}")
    print("=" * 70)
    if task.output:
        text = task.output.raw
        if len(text) > 2500:
            print(text[:2500])
            print(f"\n... [{len(text) - 2500} more characters]")
        else:
            print(text)
    else:
        print("[No output captured]")

### 6.3 Pipeline Metrics

In [ ]:
print("RETENTION PIPELINE METRICS")
print("=" * 65)
print(f"{'Phase':<10} {'Task':<25} {'Agent':<25} {'Output':>8}")
print("-" * 65)

total = 0
for phase, name, task in zip(phases, task_names, tasks):
    length = len(task.output.raw) if task.output else 0
    total += length
    agent_short = task.agent.role.split("&")[0].strip()[:23]
    print(f"{phase:<10} {name:<25} {agent_short:<25} {length:>6,}")

print("-" * 65)
print(f"{'':10} {'TOTAL':<50} {total:>6,}")
print(f"\nTicket: {COMPLAINT['ticket_id']} | Customer: {COMPLAINT['customer']}")
print(f"MRR: ${COMPLAINT['mrr']}/mo | Tenure: {COMPLAINT['tenure']}")

if hasattr(result, 'token_usage') and result.token_usage:
    print(f"Tokens: {result.token_usage.get('total_tokens', 'N/A')}")

## 7. Save Case File

In [ ]:
# Save the complete case file
case_lines = [
    f"# Case File: {COMPLAINT['ticket_id']}",
    f"**Customer:** {COMPLAINT['customer']} at {COMPLAINT['company']}",
    f"**Plan:** {COMPLAINT['plan']} | MRR: ${COMPLAINT['mrr']}/mo",
    f"**Generated:** {datetime.now().isoformat()}",
    "---",
]

section_titles = [
    "Phase 1: Complaint Analysis (DIAGNOSE)",
    "Phase 2: Churn Risk Assessment (ASSESS)",
    "Phase 3: Resolution Package (RESPOND)",
    "Phase 4: Account Summary (DOCUMENT)",
]
for title, task in zip(section_titles, tasks):
    case_lines.append(f"\n## {title}\n")
    case_lines.append(task.output.raw if task.output else "[No output]")
    case_lines.append("\n---")

case_file = "\n".join(case_lines)
with open("case_file.md", "w", encoding="utf-8") as f:
    f.write(case_file)

print(f"Case file saved: case_file.md ({len(case_file):,} chars)")

## 8. Experiment: Different Complaint (Low Severity)

Run the same crew on a low-severity, first-time complaint to observe how the collaboration flow produces a completely different response. The **same agents** with the **same backstories** will adapt their behavior based on the upstream diagnosis.

In [ ]:
# =====================================================
# SECOND COMPLAINT — Low severity, first-time
# =====================================================
COMPLAINT_2 = {
    "ticket_id": "SUP-2025-8912",
    "customer": "James Park",
    "company": "BrightLine Marketing",
    "plan": "Starter Plan ($99/mo)",
    "tenure": "3 months",
    "mrr": 99,
    "previous_tickets": 0,
    "last_ticket_topic": "N/A (first ticket)",
    "complaint_text": (
        "Hi there, I noticed that the date format in the exported CSV reports "
        "shows MM/DD/YYYY but our team uses DD/MM/YYYY (we're based in the UK). "
        "Is there a way to change the default date format in settings? I looked "
        "in the preferences but couldn't find it. Not urgent at all — we can "
        "convert manually in Excel for now — but it would save us a few minutes "
        "each week. Thanks!"
    ),
    "channel": "In-app chat",
    "timestamp": "Wednesday 2:15 PM",
}

ticket2_context = (
    f"=== SUPPORT TICKET ===\n"
    f"Ticket ID: {COMPLAINT_2['ticket_id']}\n"
    f"Customer: {COMPLAINT_2['customer']}\n"
    f"Company: {COMPLAINT_2['company']}\n"
    f"Plan: {COMPLAINT_2['plan']}\n"
    f"Tenure: {COMPLAINT_2['tenure']}\n"
    f"MRR: ${COMPLAINT_2['mrr']}/mo\n"
    f"Previous Tickets: {COMPLAINT_2['previous_tickets']} "
    f"(last: {COMPLAINT_2['last_ticket_topic']})\n"
    f"Channel: {COMPLAINT_2['channel']}\n"
    f"Timestamp: {COMPLAINT_2['timestamp']}\n\n"
    f"=== COMPLAINT TEXT ===\n"
    f"{COMPLAINT_2['complaint_text']}"
)

print("COMPLAINT 2 (Low Severity)")
print("=" * 60)
print(f"  Customer: {COMPLAINT_2['customer']} ({COMPLAINT_2['company']})")
print(f"  Plan: {COMPLAINT_2['plan']} | Tenure: {COMPLAINT_2['tenure']}")
print(f"  Prior Tickets: {COMPLAINT_2['previous_tickets']}")
print(f"\n  {COMPLAINT_2['complaint_text']}")

In [ ]:
task2_analyze = Task(
    description=(
        f"Analyze this customer complaint:\n\n{ticket2_context}\n\n"
        "Produce structured diagnostic: category, severity (S1-S4), root cause, "
        "emotional intensity, repeat flag, customer effort score, business impact."
    ),
    expected_output="Structured complaint diagnostic with all standard fields.",
    agent=complaint_analyst,
)

task2_risk = Task(
    description=(
        f"Assess churn risk for {COMPLAINT_2['customer']} at {COMPLAINT_2['company']}, "
        f"${COMPLAINT_2['mrr']}/mo, {COMPLAINT_2['tenure']} tenure.\n\n"
        "Apply risk scoring: severity points + emotion + repeat + effort + competitive "
        "mention + renewal proximity. Calculate total, assign tier, list factors."
    ),
    expected_output="Churn risk assessment with score calculation and tier.",
    agent=churn_risk_agent,
)

task2_resolution = Task(
    description=(
        f"Draft resolution for {COMPLAINT_2['customer']} calibrated to their risk tier.\n"
        f"MRR: ${COMPLAINT_2['mrr']}/mo | Tenure: {COMPLAINT_2['tenure']}\n\n"
        "Produce: (A) customer response email, (B) internal action items, "
        "(C) compensation recommendation — all calibrated to risk level."
    ),
    expected_output="Resolution package calibrated to risk tier.",
    agent=resolution_drafter,
)

task2_summary = Task(
    description=(
        f"Account summary for {COMPLAINT_2['customer']} at {COMPLAINT_2['company']}.\n"
        f"Ticket: {COMPLAINT_2['ticket_id']}\n\n"
        "Produce: case snapshot, health score, complaint pattern, financial impact, "
        "resolution status, recommendations, CRM tags."
    ),
    expected_output="360-degree account summary.",
    agent=account_summarizer,
)

tasks_2 = [task2_analyze, task2_risk, task2_resolution, task2_summary]
print(f"Complaint 2 tasks created: {len(tasks_2)}")

In [ ]:
crew2 = Crew(
    agents=agents,
    tasks=tasks_2,
    process=Process.sequential,
    verbose=True,
    memory=False,
)

print(f"Launching Crew 2 — {datetime.now().strftime('%H:%M:%S')}")
print(f"Ticket: {COMPLAINT_2['ticket_id']} (Low severity)")
print("=" * 70)

result2 = crew2.kickoff()

print(f"\nCrew 2 completed — {datetime.now().strftime('%H:%M:%S')}")

In [ ]:
print("=" * 70)
print(f"ACCOUNT SUMMARY — {COMPLAINT_2['customer']} (Low Severity)")
print("=" * 70)
print(result2.raw)

## 9. Compare Collaboration Outputs — High vs. Low Severity

In [ ]:
print("COLLABORATION OUTPUT COMPARISON — SAME AGENTS, DIFFERENT INPUTS")
print("=" * 72)
print(f"{'Phase':<10} {'Task':<25} {'High Sev':>14} {'Low Sev':>14}")
print("-" * 72)

for phase, name, t1, t2 in zip(phases, task_names, tasks, tasks_2):
    len1 = len(t1.output.raw) if t1.output else 0
    len2 = len(t2.output.raw) if t2.output else 0
    print(f"{phase:<10} {name:<25} {len1:>11,} ch {len2:>11,} ch")

total1 = sum(len(t.output.raw) for t in tasks if t.output)
total2 = sum(len(t.output.raw) for t in tasks_2 if t.output)
print("-" * 72)
print(f"{'':10} {'TOTAL':<25} {total1:>11,} ch {total2:>11,} ch")

print(f"\nHigh severity: {COMPLAINT['customer']} — ${COMPLAINT['mrr']}/mo, repeat issue, angry")
print(f"Low severity:  {COMPLAINT_2['customer']} — ${COMPLAINT_2['mrr']}/mo, first ticket, calm")
print(f"\nKey observation: Same 4 agents produced different response intensities")
print(f"because the collaboration flow adapted to upstream findings.")

## 10. Advanced: Explicit Context for Parallel Triage

### Alternative Collaboration Pattern: Parallel Assessment

In high-volume support operations, the Complaint Analyst and Churn Risk Agent can work **independently** from the raw ticket, then the Resolution Drafter reconciles both assessments.

```
               ┌──> Complaint Analyst ──┐
Raw Ticket ────┤                        ├──> Resolution Drafter ──> Account Summarizer
               └──> Churn Risk Agent  ──┘
```

**Trade-off:** Parallel triage is faster but the Churn Risk Agent must infer severity from the raw complaint rather than the Analyst's structured diagnosis. This can result in risk scores that don't align with the complaint classification.

**When to use each pattern:**

| Pattern | Best For |
|---------|----------|
| Sequential (this notebook) | Quality-critical cases, high-value accounts, escalations |
| Parallel triage | High-volume queues where speed matters more than precision |

In [ ]:
# Parallel triage with explicit context

par_analyze = Task(
    description=(
        f"Analyze this complaint:\n\n{ticket_context}\n\n"
        "Produce structured diagnostic: category, severity, root cause, emotion, "
        "repeat flag, effort score."
    ),
    expected_output="Complaint diagnostic.",
    agent=complaint_analyst,
)

par_risk = Task(
    description=(
        f"Assess churn risk INDEPENDENTLY from the raw ticket (do NOT wait for "
        f"the complaint analysis — assess risk directly from the customer's words):\n\n"
        f"{ticket_context}\n\n"
        f"Account: ${COMPLAINT['mrr']}/mo, {COMPLAINT['tenure']} tenure.\n"
        "Score risk using your framework and assign a tier."
    ),
    expected_output="Independent churn risk assessment.",
    agent=churn_risk_agent,
    context=[],  # Empty = independent, does NOT see complaint analysis
)

# Resolution drafter reconciles both independent assessments
par_resolution = Task(
    description=(
        f"Draft resolution for {COMPLAINT['customer']}.\n\n"
        "IMPORTANT: You have received TWO independent assessments — one from the "
        "Complaint Analyst and one from the Churn Risk Agent. They worked independently "
        "and may not fully agree. Your job is to:\n"
        "1. Reconcile any differences between their assessments\n"
        "2. Use the higher risk tier if they disagree (err on the side of retention)\n"
        "3. Draft the resolution calibrated to the reconciled risk level\n"
    ),
    expected_output="Resolution package reconciling both assessments.",
    agent=resolution_drafter,
    context=[par_analyze, par_risk],  # Sees both
)

par_summary = Task(
    description=(
        f"Account summary for {COMPLAINT['customer']} integrating all prior outputs."
    ),
    expected_output="Account summary.",
    agent=account_summarizer,
    context=[par_analyze, par_risk, par_resolution],
)

par_tasks = [par_analyze, par_risk, par_resolution, par_summary]

print("PARALLEL TRIAGE FLOW")
print("=" * 60)
print("              ┌──> Complaint Analyst ──┐")
print("  Raw Ticket ─┤                        ├──> Resolution ──> Summary")
print("              └──> Churn Risk (indep.) ┘")
print()
par_names = ["Complaint Analysis", "Risk (independent)", "Resolution (reconciled)", "Account Summary"]
for i, (name, task) in enumerate(zip(par_names, par_tasks), 1):
    ctx = task.context
    if ctx is None:
        ctx_str = "auto (all prior)"
    elif len(ctx) == 0:
        ctx_str = "NONE (independent)"
    else:
        ctx_str = " + ".join(t.agent.role.split("&")[0].strip()[:22] for t in ctx)
    print(f"  Task {i}: {name}")
    print(f"    Context: {ctx_str}")

In [ ]:
crew_par = Crew(
    agents=agents,
    tasks=par_tasks,
    process=Process.sequential,
    verbose=True,
    memory=False,
)

print(f"Launching parallel-triage crew — {datetime.now().strftime('%H:%M:%S')}")
result_par = crew_par.kickoff()
print(f"\nParallel-triage crew completed — {datetime.now().strftime('%H:%M:%S')}")

In [ ]:
print("PARALLEL TRIAGE RESULTS")
print("=" * 65)
for name, task in zip(par_names, par_tasks):
    length = len(task.output.raw) if task.output else 0
    ctx = task.context
    if ctx is None:
        ctx_str = "auto"
    elif len(ctx) == 0:
        ctx_str = "independent"
    else:
        ctx_str = f"{len(ctx)} inputs"
    print(f"  {name:<30} {length:>6,} chars | Context: {ctx_str}")

## 11. Key Takeaways

### What We Built
- A **4-agent customer retention pipeline** (Diagnose → Assess → Respond → Document)
- Ran it on **two complaints** (high-severity repeat issue + low-severity first-time request) to demonstrate adaptive collaboration
- Demonstrated **parallel triage** where analyst and risk agent work independently and the resolver reconciles

### Collaboration Flow Concepts
1. **Adaptive collaboration** — Each agent changes its behavior based on upstream findings, not just its own instructions. The same Resolution Drafter produces an executive-level response for critical risk and a friendly template for low risk
2. **Collaboration contracts** — Each agent's output contains structured fields that downstream agents consume as inputs. The Complaint Analyst's severity (S1-S4) maps directly to the Churn Risk Agent's point system
3. **Data dependencies determine flow** — Sequential execution is required when agents have true data dependencies (risk scoring requires diagnosis). Parallel execution works only for independent assessments
4. **Context accumulation vs. selective context** — Sequential gives each agent ALL prior outputs; explicit `context` lets you control exactly which outputs each agent sees

### Agent Design Lessons for Collaboration
- **Complaint Analyst:** Define output fields as an explicit schema (category enum, severity scale, emotion enum) so downstream agents can consume them reliably
- **Churn Risk Agent:** Use a quantitative scoring framework (point system) so the risk calculation is transparent and the Resolution Drafter knows exactly WHY the risk is at a given level
- **Resolution Drafter:** Build the response-intensity mapping (risk tier → tone + compensation) into the backstory, not just the task — it produces more consistent calibration
- **Account Summarizer:** Design as an archival document — it should be readable months later by someone who never saw the original complaint

### Production Tips
- Connect to a real CRM (Salesforce, HubSpot) to pull actual account history and ticket records
- Use `output_pydantic` with typed schemas to enforce the collaboration contract between agents
- Enable `memory=True` to build up customer context across multiple complaints from the same account
- Add a **Sentiment Trend Agent** that tracks emotional intensity across a customer's ticket history
- Consider a **Proactive Outreach Agent** that generates check-in scripts for at-risk accounts before renewal